In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

DATA_INTERIM.mkdir(parents=True, exist_ok=True)

In [3]:
usda = pd.read_csv(DATA_RAW / "StateAndCountyData.csv")

cdc_files = list(DATA_RAW.glob("PLACES*.csv"))
cdc = pd.read_csv(cdc_files[0])

print("USDA:", usda.shape)
print("CDC:", cdc.shape)

USDA: (957753, 5)
CDC: (3144, 167)


In [4]:
selected_predictors = [
    "PCT_LACCESS_POP19",
    "PCT_LACCESS_LOWI19",
    "GROCPTH20",
    "CONVSPTH20",
    "FFRPTH20",
    "FSRPTH20",
    "MEDHHINC21",
    "POVRATE21",
    "CHILDPOVRATE21",
    "DEEPPOVRATE21",
    "PC_SNAPBEN22",
    "PCT_65OLDER20",
    "PCT_18YOUNGER20",
    "PCT_NHWHITE20",
    "PCT_NHBLACK20",
    "PCT_HISP20",
    "PCT_NHASIAN20",
    "RECFACPTH20"
]

In [5]:
usda_selected = usda[
    usda["Variable_Code"].isin(selected_predictors)
].copy()

print("Selected USDA rows:", len(usda_selected))
print("Unique selected variables:", usda_selected["Variable_Code"].nunique())

Selected USDA rows: 56685
Unique selected variables: 18


In [6]:
usda_selected["FIPS"] = (
    usda_selected["FIPS"]
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)

print(usda_selected["FIPS"].dtype)
display(
    usda_selected[
        ["FIPS", "State", "County", "Variable_Code", "Value"]
    ].head()
)

string


,FIPS,State,County,Variable_Code,Value
4,01001,AL,Autauga,PCT_LACCESS_POP19,33.906700
9,01001,AL,Autauga,PCT_LACCESS_LOWI19,13.020998
69,01003,AL,Baldwin,PCT_LACCESS_POP19,25.122238
74,01003,AL,Baldwin,PCT_LACCESS_LOWI19,7.936779
134,01005,AL,Barbour,PCT_LACCESS_POP19,20.520679


In [7]:
cdc["CountyFIPS"] = (
    cdc["CountyFIPS"]
    .astype("string")
    .str.zfill(5)
)

print(cdc["CountyFIPS"].dtype)

display(
    cdc[
        ["CountyFIPS", "StateAbbr", "CountyName", "OBESITY_AdjPrev"]
    ].head(10)
)

string


,CountyFIPS,StateAbbr,CountyName,OBESITY_AdjPrev
0,17065,IL,Hamilton,37.8
1,36099,NY,Seneca,37.4
2,40061,OK,Haskell,43.7
3,17097,IL,Lake,31.9
4,20005,KS,Atchison,40.5
5,02020,AK,Anchorage,29.6
6,40133,OK,Seminole,43.0
7,54055,WV,Mercer,42.5
8,27013,MN,Blue Earth,42.1
9,31019,NE,Buffalo,43.4


In [8]:
print("USDA unique FIPS:", usda_selected["FIPS"].nunique())
print("CDC unique FIPS:", cdc["CountyFIPS"].nunique())

print(
    "USDA FIPS not 5 characters:",
    (usda_selected["FIPS"].str.len() != 5).sum()
)

print(
    "CDC FIPS not 5 characters:",
    (cdc["CountyFIPS"].str.len() != 5).sum()
)

USDA unique FIPS: 3156
CDC unique FIPS: 3144
USDA FIPS not 5 characters: 0
CDC FIPS not 5 characters: 0


In [9]:
fips_with_minus8888 = (
    usda_selected.loc[
        usda_selected["Value"] == -8888,
        "FIPS"
    ]
    .dropna()
    .unique()
)

print("Counties with -8888:", len(fips_with_minus8888))
print(fips_with_minus8888)

Counties with -8888: 13
<StringArray>
['02261', '02270', '46113', '51515', '09110', '09120', '09130', '09140',
 '09150', '09160', '09170', '09180', '09190']
Length: 13, dtype: string


In [10]:
usda_selected_clean = usda_selected[
    ~usda_selected["FIPS"].isin(fips_with_minus8888)
].copy()

print("Rows before -8888 exclusion:", len(usda_selected))
print("Rows after -8888 exclusion:", len(usda_selected_clean))

print(
    "Remaining -8888 values:",
    (usda_selected_clean["Value"] == -8888).sum()
)

Rows before -8888 exclusion: 56685
Rows after -8888 exclusion: 56574
Remaining -8888 values: 0


In [11]:
usda_selected_clean["Value"] = usda_selected_clean["Value"].replace(
    -9999,
    np.nan
)

print(
    "Remaining -9999 values:",
    (usda_selected_clean["Value"] == -9999).sum()
)

print(
    "Missing values after conversion:",
    usda_selected_clean["Value"].isna().sum()
)

Remaining -9999 values: 0
Missing values after conversion: 3855


In [12]:
county_info = (
    usda_selected_clean[
        ["FIPS", "State", "County"]
    ]
    .drop_duplicates(subset="FIPS")
    .copy()
)

print("County info rows:", len(county_info))
print("Duplicate FIPS:", county_info["FIPS"].duplicated().sum())

County info rows: 3143
Duplicate FIPS: 0


In [13]:
usda_wide = (
    usda_selected_clean
    .pivot(
        index="FIPS",
        columns="Variable_Code",
        values="Value"
    )
    .reset_index()
)

usda_wide.columns.name = None

print("USDA wide shape:", usda_wide.shape)
display(usda_wide.head())

USDA wide shape: (3143, 19)


,FIPS,CHILDPOVRATE21,CONVSPTH20,DEEPPOVRATE21,FFRPTH20,FSRPTH20,GROCPTH20,MEDHHINC21,PCT_18YOUNGER20,PCT_65OLDER20,PCT_HISP20,PCT_LACCESS_LOWI19,PCT_LACCESS_POP19,PCT_NHASIAN20,PCT_NHBLACK20,PCT_NHWHITE20,PC_SNAPBEN22,POVRATE21,RECFACPTH20
0,01001,16.1,0.463087,6.250216,0.801496,0.569953,0.071244,66444.0,24.287050,15.718051,3.600034,13.020998,33.906700,1.484568,19.304481,70.711674,36.285511,10.7,0.160299
1,01003,16.4,0.514639,4.043401,0.750152,1.072891,0.126479,65658.0,21.269637,21.871103,5.473601,7.936779,25.122238,0.875448,7.766852,80.466589,23.936159,10.8,0.095950
2,01005,35.1,0.610029,12.826966,0.976046,0.528692,0.203343,38649.0,20.176030,20.231535,5.986600,10.433171,20.520679,0.408357,46.980930,43.951949,62.904144,23.0,NaN
3,01007,29.0,0.451753,9.076190,0.316227,0.316227,0.180701,48454.0,21.351994,16.220338,3.319428,0.445866,1.593457,0.116629,19.692280,73.754093,39.449314,20.6,NaN
4,01009,16.7,0.501045,5.291223,0.414658,0.293716,0.069110,56894.0,23.220144,18.625495,9.759191,2.512206,6.807624,0.294247,1.396828,84.154632,28.657486,12.0,0.069110


In [14]:
usda_wide = county_info.merge(
    usda_wide,
    on="FIPS",
    how="inner"
)

print("Final USDA wide shape:", usda_wide.shape)
print("Duplicate FIPS:", usda_wide["FIPS"].duplicated().sum())

display(usda_wide.head())

Final USDA wide shape: (3143, 21)
Duplicate FIPS: 0


,FIPS,State,County,CHILDPOVRATE21,CONVSPTH20,DEEPPOVRATE21,FFRPTH20,FSRPTH20,GROCPTH20,MEDHHINC21,...,PCT_65OLDER20,PCT_HISP20,PCT_LACCESS_LOWI19,PCT_LACCESS_POP19,PCT_NHASIAN20,PCT_NHBLACK20,PCT_NHWHITE20,PC_SNAPBEN22,POVRATE21,RECFACPTH20
0,01001,AL,Autauga,16.1,0.463087,6.250216,0.801496,0.569953,0.071244,66444.0,...,15.718051,3.600034,13.020998,33.906700,1.484568,19.304481,70.711674,36.285511,10.7,0.160299
1,01003,AL,Baldwin,16.4,0.514639,4.043401,0.750152,1.072891,0.126479,65658.0,...,21.871103,5.473601,7.936779,25.122238,0.875448,7.766852,80.466589,23.936159,10.8,0.095950
2,01005,AL,Barbour,35.1,0.610029,12.826966,0.976046,0.528692,0.203343,38649.0,...,20.231535,5.986600,10.433171,20.520679,0.408357,46.980930,43.951949,62.904144,23.0,NaN
3,01007,AL,Bibb,29.0,0.451753,9.076190,0.316227,0.316227,0.180701,48454.0,...,16.220338,3.319428,0.445866,1.593457,0.116629,19.692280,73.754093,39.449314,20.6,NaN
4,01009,AL,Blount,16.7,0.501045,5.291223,0.414658,0.293716,0.069110,56894.0,...,18.625495,9.759191,2.512206,6.807624,0.294247,1.396828,84.154632,28.657486,12.0,0.069110


In [15]:
print("Rows:", len(usda_wide))
print("Unique FIPS:", usda_wide["FIPS"].nunique())
print("Duplicate FIPS:", usda_wide["FIPS"].duplicated().sum())

print("\nPredictors present:")
print(sum(col in usda_wide.columns for col in selected_predictors), "of 18")

print("\nTotal missing predictor values:")
print(usda_wide[selected_predictors].isna().sum().sum())

Rows: 3143
Unique FIPS: 3143
Duplicate FIPS: 0

Predictors present:
18 of 18

Total missing predictor values:
3855


In [16]:
wide_missingness = pd.DataFrame({
    "missing_count": usda_wide[selected_predictors].isna().sum(),
    "missing_pct": (
        usda_wide[selected_predictors].isna().mean() * 100
    )
}).round(2)

display(wide_missingness)

,missing_count,missing_pct
PCT_LACCESS_POP19,2,0.06
PCT_LACCESS_LOWI19,2,0.06
GROCPTH20,909,28.92
CONVSPTH20,284,9.04
FFRPTH20,471,14.99
FSRPTH20,282,8.97
MEDHHINC21,1,0.03
POVRATE21,1,0.03
CHILDPOVRATE21,1,0.03
DEEPPOVRATE21,0,0.00


In [17]:
cdc_target = cdc[
    [
        "CountyFIPS",
        "StateAbbr",
        "CountyName",
        "OBESITY_AdjPrev"
    ]
].copy()

cdc_target = cdc_target.rename(
    columns={"CountyFIPS": "FIPS"}
)

print("CDC target rows:", len(cdc_target))
print("Unique FIPS:", cdc_target["FIPS"].nunique())

CDC target rows: 3144
Unique FIPS: 3144


In [18]:
usda_fips = set(usda_wide["FIPS"])
cdc_fips = set(cdc_target["FIPS"])

matched_fips = usda_fips & cdc_fips
usda_only_fips = usda_fips - cdc_fips
cdc_only_fips = cdc_fips - usda_fips

print("USDA counties:", len(usda_fips))
print("CDC counties:", len(cdc_fips))
print("Matched counties:", len(matched_fips))
print("USDA only:", len(usda_only_fips))
print("CDC only:", len(cdc_only_fips))

USDA counties: 3143
CDC counties: 3144
Matched counties: 3135
USDA only: 8
CDC only: 9


In [19]:
usda_only = usda_wide[
    usda_wide["FIPS"].isin(usda_only_fips)
][["FIPS", "State", "County"]]

cdc_only = cdc_target[
    cdc_target["FIPS"].isin(cdc_only_fips)
][["FIPS", "StateAbbr", "CountyName"]]

print("USDA-only counties:")
display(usda_only)

print("CDC-only counties:")
display(cdc_only)

USDA-only counties:


,FIPS,State,County
309,09001,CT,Fairfield
310,09003,CT,Hartford
311,09005,CT,Litchfield
312,09007,CT,Middlesex
313,09009,CT,New Haven
314,09011,CT,New London
315,09013,CT,Tolland
316,09015,CT,Windham


CDC-only counties:


,FIPS,StateAbbr,CountyName
543,09180,CT,Southeastern Connecticut
744,09140,CT,Naugatuck Valley
1080,09190,CT,Western Connecticut
1404,09130,CT,Lower Connecticut River Valley
1655,09150,CT,Northeastern Connecticut
2257,09120,CT,Greater Bridgeport
2434,09160,CT,Northwest Hills
2775,09170,CT,South Central Connecticut
3081,09110,CT,Capitol


In [20]:
display(usda_only)

,FIPS,State,County
309,09001,CT,Fairfield
310,09003,CT,Hartford
311,09005,CT,Litchfield
312,09007,CT,Middlesex
313,09009,CT,New Haven
314,09011,CT,New London
315,09013,CT,Tolland
316,09015,CT,Windham


In [21]:
modeling_data = usda_wide.merge(
    cdc_target[
        ["FIPS", "OBESITY_AdjPrev"]
    ],
    on="FIPS",
    how="inner",
    validate="one_to_one"
)

print("Integrated dataset shape:", modeling_data.shape)
print("Unique FIPS:", modeling_data["FIPS"].nunique())
print("Duplicate FIPS:", modeling_data["FIPS"].duplicated().sum())
print("Missing target values:", modeling_data["OBESITY_AdjPrev"].isna().sum())

display(modeling_data.head())

Integrated dataset shape: (3135, 22)
Unique FIPS: 3135
Duplicate FIPS: 0
Missing target values: 0


,FIPS,State,County,CHILDPOVRATE21,CONVSPTH20,DEEPPOVRATE21,FFRPTH20,FSRPTH20,GROCPTH20,MEDHHINC21,...,PCT_HISP20,PCT_LACCESS_LOWI19,PCT_LACCESS_POP19,PCT_NHASIAN20,PCT_NHBLACK20,PCT_NHWHITE20,PC_SNAPBEN22,POVRATE21,RECFACPTH20,OBESITY_AdjPrev
0,01001,AL,Autauga,16.1,0.463087,6.250216,0.801496,0.569953,0.071244,66444.0,...,3.600034,13.020998,33.906700,1.484568,19.304481,70.711674,36.285511,10.7,0.160299,38.4
1,01003,AL,Baldwin,16.4,0.514639,4.043401,0.750152,1.072891,0.126479,65658.0,...,5.473601,7.936779,25.122238,0.875448,7.766852,80.466589,23.936159,10.8,0.095950,36.8
2,01005,AL,Barbour,35.1,0.610029,12.826966,0.976046,0.528692,0.203343,38649.0,...,5.986600,10.433171,20.520679,0.408357,46.980930,43.951949,62.904144,23.0,NaN,43.8
3,01007,AL,Bibb,29.0,0.451753,9.076190,0.316227,0.316227,0.180701,48454.0,...,3.319428,0.445866,1.593457,0.116629,19.692280,73.754093,39.449314,20.6,NaN,41.4
4,01009,AL,Blount,16.7,0.501045,5.291223,0.414658,0.293716,0.069110,56894.0,...,9.759191,2.512206,6.807624,0.294247,1.396828,84.154632,28.657486,12.0,0.069110,37.3


In [22]:
cdc_names = cdc_target[
    ["FIPS", "StateAbbr", "CountyName"]
]

name_check = modeling_data[
    ["FIPS", "State", "County"]
].merge(
    cdc_names,
    on="FIPS",
    how="left",
    validate="one_to_one"
)

display(name_check.head(20))

,FIPS,State,County,StateAbbr,CountyName
0,01001,AL,Autauga,AL,Autauga
1,01003,AL,Baldwin,AL,Baldwin
2,01005,AL,Barbour,AL,Barbour
3,01007,AL,Bibb,AL,Bibb
4,01009,AL,Blount,AL,Blount
5,01011,AL,Bullock,AL,Bullock
6,01013,AL,Butler,AL,Butler
7,01015,AL,Calhoun,AL,Calhoun
8,01017,AL,Chambers,AL,Chambers
9,01019,AL,Cherokee,AL,Cherokee


In [23]:
state_mismatches = name_check[
    name_check["State"] != name_check["StateAbbr"]
]

print("State mismatches:", len(state_mismatches))
display(state_mismatches)

State mismatches: 0


,FIPS,State,County,StateAbbr,CountyName


In [24]:
county_name_mismatches = name_check[
    name_check["County"] != name_check["CountyName"]
]

print("County name mismatches:", len(county_name_mismatches))
display(county_name_mismatches.head(30))

County name mismatches: 0


,FIPS,State,County,StateAbbr,CountyName


In [25]:
print("Total missing predictor values:",
      modeling_data[selected_predictors].isna().sum().sum())

print("Counties with at least one missing predictor:",
      modeling_data[selected_predictors].isna().any(axis=1).sum())

print("Counties with complete predictor data:",
      modeling_data[selected_predictors].notna().all(axis=1).sum())

Total missing predictor values: 3855
Counties with at least one missing predictor: 1911
Counties with complete predictor data: 1224


In [26]:
output_path = DATA_INTERIM / "modeling_data.csv"

modeling_data.to_csv(
    output_path,
    index=False
)

print("Saved to:", output_path)
print("Shape:", modeling_data.shape)

Saved to: /Users/chelsearose/Documents/GitHub/county-obesity-prediction/data/interim/modeling_data.csv
Shape: (3135, 22)


In [27]:
pd.read_csv("modeling_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'modeling_data.csv'

In [28]:
test_reload = pd.read_csv(
    output_path,
    dtype={"FIPS": "string"}
)

print(test_reload["FIPS"].head())
print("FIPS dtype:", test_reload["FIPS"].dtype)
print("Invalid FIPS lengths:", (test_reload["FIPS"].str.len() != 5).sum())
print("Reloaded shape:", test_reload.shape)

0    01001
1    01003
2    01005
3    01007
4    01009
Name: FIPS, dtype: string
FIPS dtype: string
Invalid FIPS lengths: 0
Reloaded shape: (3135, 22)
